<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c35-header.png" alt="Nextia Learning · Forecasting and Time Series: Predict What Comes Next" width="100%">

# Solution: Meet the Larkfield data

**[Meet the Larkfield data](https://learning.nextia-ai.com/courses/forecasting/m01/meet-the-larkfield-data/)** · Forecasting and Time Series: Predict What Comes Next · Module 1, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** open the course data, read its tables, draw one product with its promotions and stock-outs, and run the backtest that every module of the course fills.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | Why a forecast must start from a decision, from [Frame the forecast](https://learning.nextia-ai.com/courses/forecasting/m01/frame-the-forecast/). How to read a pandas table. |
| **Tested** | Python 3.12 with pandas 2.3.3, statsforecast 2.1.1, mlforecast 1.1.0, hierarchicalforecast 1.5.3, LightGBM 4.7.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m01/meet-the-larkfield-data/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C35/M01-L03-meet-the-larkfield-data/meet-the-larkfield-data-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 2.3.3, statsforecast 2.1.1, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==2.3.3 statsforecast==2.1.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import statsforecast; print("statsforecast", statsforecast.__version__)

## Setup: get the practice data

Run the next cell. It downloads the Larkfield data of this course (about 2 MB) into a folder `larkfield-forecasting/data`, and the course's backtest helper `backtest.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: 403758 rows of daily sales (200 products, 3 regions), from 2024-04-01 to 2026-03-29.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-forecasting` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: we made it for this course. It has no real people, products or prices.

In [ ]:
# Setup: download the course data into larkfield-forecasting/data/ and check it.
import gzip, hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C35/"
FILES = {
    "calendar.csv": "a5d1d5a3b640a983de72a30fca0e627a4ef23e797b1d80619e9ffcaadf118844",
    "demand.csv.gz": "8b39303985cbdb9bb1f3c705cd927ed6aaeb6f9665bd56ad4c3f8cb2b8578c99",
    "orders.csv": "bd8059fd73fe294daa157f8a6b6ae3f58319fa644eaa7ff0d31122b02cf05e30",
    "price_changes.csv": "889b258be4e933b87065d81d0771d6cd9b630ee90223b0a6ab074129c42153cb",
    "products.csv": "198b0a3de6eea8128af68f3cbc3a5c6acd3a64ac854fff18d37660857be8cfbf",
    "promotions.csv": "ca147959faa4c36c757742de7121976ffe79d59b53049dbbc8337a7547541f4a",
    "roster_hourly.csv": "6c2e0e96881b36ded52ed19458515c71f025288ddaebd589b50c82a8ad7fa69b",
    "tickets_daily.csv": "36f7067db060fdd2bd084c408c0622370a012773849f243020336f1fc678b31d",
    "tickets_hourly.csv.gz": "43f7fbb3e360a8e4a0cb4e101ea6a6242a76f0c37204d76ee9413ea3adbedd92",
    "weather.csv": "43a199a8b235940e291195bfd026901b62a10189b29f3a3ad9527b5382544276",
    "backtest.py": "764ecd38c6e96632be6f0e139e05eadd7b07d14a4ece73d881fc7559c2fa203a",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-forecasting").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and sha256(path) == expected:
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-forecasting and run again.")

for name, expected in FILES.items():
    if name == "backtest.py":
        fetch(name, LABS + name, PROJECT / name, expected)
    else:
        fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
with gzip.open("data/demand.csv.gz", "rt") as f:
    header = f.readline()
    rows = [line.split(",", 3)[:3] for line in f]
print(f"Ready: {len(rows)} rows of daily sales ({len({r[2] for r in rows})} products, {len({r[1] for r in rows})} regions), "
      f"from {min(r[0] for r in rows)} to {max(r[0] for r in rows)}.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course's backtest helper as `bt`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the backtest helper, and checks that do not show the answer.
import hashlib, json, logging, os, warnings
os.environ["NIXTLA_ID_AS_COL"] = "1"
import numpy as np
import pandas as pd
import backtest as bt
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")
logging.getLogger("lightning").setLevel(logging.ERROR)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def load_sales(products=None):
    """Daily sales per product, the three regions added up, in the long format of the backtest:
    unique_id (the product), ds (the day), y (units sold), price, on_promotion, stock_out.
    The missing day 2025-10-19 gets the mean of the same weekday one week before and after."""
    d = pd.read_csv("data/demand.csv.gz", parse_dates=["date"])
    if products is not None:
        d = d[d["product_id"].isin(products)]
    g = d.groupby(["product_id", "date"], as_index=False).agg(
        y=("sales", "sum"), price=("price", "mean"), on_promotion=("on_promotion", "max"), stock_out=("stock_out", "max"))
    out = []
    for uid, s in g.rename(columns={"product_id": "unique_id", "date": "ds"}).groupby("unique_id"):
        s = s.set_index("ds").reindex(pd.date_range(s["ds"].min(), s["ds"].max(), freq="D"))
        gap = s["y"].isna()
        before, after = s["y"].shift(7), s["y"].shift(-7)
        s.loc[gap, "y"] = ((before + after) / 2).fillna(before).fillna(after)[gap].round()
        s = s.ffill()
        s["unique_id"] = uid
        out.append(s.rename_axis("ds").reset_index())
    return pd.concat(out, ignore_index=True)[["unique_id", "ds", "y", "price", "on_promotion", "stock_out"]]

# The course backtest (the same in every module): 13 origins, 28 days apart, the last one 2026-03-15.
ORIGINS = bt.origins("2026-03-15", n=13, every="28D")
BACKTEST_PRODUCTS = ['P001', 'P002', 'P005', 'P006', 'P009', 'P013', 'P018', 'P024', 'P025', 'P029', 'P032', 'P039', 'P045', 'P057', 'P068', 'P075', 'P077', 'P078', 'P080', 'P081', 'P087', 'P088', 'P093', 'P095', 'P108', 'P110', 'P121', 'P130', 'P133', 'P136', 'P137', 'P141', 'P145', 'P147', 'P148', 'P156', 'P158', 'P159', 'P166', 'P169', 'P172', 'P174', 'P176', 'P183', 'P184', 'P186', 'P191', 'P192', 'P194', 'P199']
PRICES = pd.read_csv("data/products.csv").set_index("product_id")["base_price"].to_dict()

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The tables

Larkfield's data has ten tables. Each one has a date (or an hour) in every row, because a **time series** is a list of values in time order. The next cell reads all of them and prints their size and columns.

> **Predict.** Which table has the most rows: the daily sales of each product in each region, or the hourly help-desk tickets? Write your prediction, then run the cell.

In [ ]:
names = ["demand.csv.gz", "orders.csv", "tickets_daily.csv", "tickets_hourly.csv.gz", "roster_hourly.csv",
         "weather.csv", "calendar.csv", "products.csv", "promotions.csv", "price_changes.csv"]
tables = {name: pd.read_csv(f"data/{name}") for name in names}
for name, t in tables.items():
    print(f"{name:22} {len(t):7} rows  {list(t.columns)}")

> **Check.** You should see `demand.csv.gz` with `403758 rows`, `tickets_hourly.csv.gz` with `52416 rows` and `products.csv` with `200 rows`.

- **demand**: one row per product, region and day: the price, a promotion flag, the units **sold** and a **stock_out** flag.
- **orders** and **tickets_daily**: one row per channel and day. **tickets_hourly** and **roster_hourly**: one row per hour.
- **calendar**, **promotions** and **price_changes** go 13 weeks past the last sales day: a shop plans them in advance, so a forecast may use them.

The history runs from 1 April 2024 to 29 March 2026: 104 weeks. The 13 weeks after it are the **hidden future**: we keep them back to score your final forecasts, as reality would.

## 2. One product: the parasol

Kenji's most discussed product is **P025, Parasol, 3 m, sand**. The next cell adds up its three regions per day and draws two years of sales. It marks the promotion days and the days when a region had no stock.

In [ ]:
import matplotlib.pyplot as plt

demand = pd.read_csv("data/demand.csv.gz", parse_dates=["date"])
parasol = demand[demand["product_id"] == "P025"].groupby("date").agg(
    sales=("sales", "sum"), promotion=("on_promotion", "max"), stock_out=("stock_out", "max"))
print(len(parasol), "days,", int(parasol["sales"].sum()), "parasols sold,",
      int(parasol["promotion"].sum()), "promotion days,", int(parasol["stock_out"].sum()), "days with a stock-out")

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(parasol.index, parasol["sales"], lw=0.8, label="units sold per day")
promo = parasol[parasol["promotion"] == 1]
ax.scatter(promo.index, promo["sales"], color="tab:green", s=12, label="promotion day", zorder=3)
out = parasol[parasol["stock_out"] == 1]
ax.scatter(out.index, out["sales"], color="tab:red", marker="x", s=25, label="a region had no stock", zorder=3)
ax.set_xlabel("Day")
ax.set_ylabel("Parasols sold")
ax.legend()
plt.show()

> **Check.** You should see `727 days, 4201 parasols sold, 14 promotion days, 9 days with a stock-out`. The chart shows two summers. The second summer is lower, with one tall spike at the start of July.

727 days, not 728: one day is missing. We find it in section 4.

> **Your turn.** Product **P002** (Folding chair, black) had a long stock-out in the **north** region. Count the days with `stock_out == 1` for P002 in the north. Put the number in `p002_north_out` as a Python integer (`int(...)`). Then run the check cell.

In [ ]:
rows = demand[(demand["product_id"] == "P002") & (demand["region"] == "north")]
p002_north_out = int(rows["stock_out"].sum())
print(p002_north_out)

In [ ]:
expect('p002_north_out', p002_north_out, 9)

Nine days in a row, from 12 to 20 May 2025, with zero sales. Customers still wanted folding chairs on those days: the shelf was empty. **Sales are not demand.** Module 2 shows what a forecast learns if you train it on those zeros.

## 3. The backtest table

Every module of this course adds rows to one table. Each row is one forecasting method, tested the same way on the same data:

- **the same series**: 50 products (`BACKTEST_PRODUCTS`), their three regions added up;
- **the same test**: 13 cut-off days, 28 days apart, from 13 April 2025 to 15 March 2026. At each cut-off, the method sees only the days before it and forecasts the next 14 days;
- **the same measures**: MASE, WAPE, bias, pinball loss, fit time and the cost of the stock order. Module 3 explains each one.

The helper `backtest.py` does this for any method. The next cell runs the simplest method, "the same weekday last week", and prints the first row of the table.

> **Predict.** A forecast that says "the same as last week" misses by how much, in percent of the units sold: about 10%, 25% or 50%? Then run the cell.

In [ ]:
sales = load_sales(BACKTEST_PRODUCTS)
fc = bt.backtest(sales, bt.seasonal_naive(7), h=14, origins=ORIGINS)
row = bt.score(fc, sales, "seasonal naive", season=7, cost=bt.stock_cost(PRICES))
table = bt.show([row])

> **Check.** You should see one row: `seasonal naive`, horizon `14`, MASE `1.034`, WAPE % `51.3`, bias % `-0.5` and cost $ `16739`.

A WAPE of 51% means: the absolute misses add up to half of all units sold. Daily sales of one product are noisy. The bias of −0.5% says the misses are not mostly too high or too low. The cost is the cost of Kenji's orders for 14 days, on average per cut-off day, if he orders this forecast (Module 7 explains it).

## 4. Failure case: a day that is not there

Mei, the data engineer, says the orders feed is complete. The next cell counts the days in `orders.csv` and the days between its first and its last date.

In [ ]:
orders = pd.read_csv("data/orders.csv", parse_dates=["date"])
per_day = orders.groupby("date")["orders"].sum()
all_days = pd.date_range(per_day.index.min(), per_day.index.max(), freq="D")
print(len(per_day), "days with orders,", len(all_days), "days in the calendar")
print("missing:", [d.date().isoformat() for d in all_days.difference(per_day.index)])

> **Check.** You should see `727 days with orders, 728 days in the calendar` and `missing: ['2025-10-19']`.

The feed dropped one Sunday. A model that expects one row per day would quietly put Saturday next to Monday. **Always reindex a series to every period before you forecast**, and decide what the missing day gets. In this course, the helper `load_sales()` gives it the mean of the same weekday one week before and one week after. Module 2 discusses this choice.

## Next

You have opened every table, seen sales that are not demand, run the first row of the backtest table and found a missing day. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m01/meet-the-larkfield-data/) has the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Trend, season and noise](https://learning.nextia-ai.com/courses/forecasting/m02/trend-season-and-noise/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/forecasting/m01/meet-the-larkfield-data/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.